# 🔎 Busca em Largura (BFS)

A **Busca em Largura** (Breadth-First Search - BFS) é um dos algoritmos fundamentais em grafos. Dado um grafo $G=(V,E)$ e um vértice de origem $s$, a BFS explora as arestas de $G$ para descobrir todos os vértices acessíveis a partir de $s$, computando distâncias mínimas em número de arestas e gerando a **árvore de primeira extensão** (árvore breadth-first) com raiz em $s$.

## 🎯 Objetivos e Resultados
1. Explorar o grafo a partir de $s$ e descobrir todos os vértices acessíveis.
2. Calcular a **distância** $d[v]$ (em número de arestas) de $s$ até cada vértice acessível $v$.
3. Garantir que o caminho na **árvore BFS** de $s$ até $v$ é um caminho mais curto (mínimo número de arestas).
4. Produzir uma **árvore breadth-first** enraizada em $s$ contendo todos os vértices alcançáveis.

## 🧠 Ideia e Estratégia
A BFS visita vértices por **camadas** (níveis): primeiro distância 0 (o próprio $s$), depois todos na distância 1, depois todos na distância 2, e assim por diante. Para isso, utiliza-se uma **fila (queue)** que garante ordem FIFO.

## 🎨 Cores e Estados dos Vértices
- Branco: não descoberto (inicial).
- Cinza: descoberto e enfileirado (na fronteira).
- Preto: finalizado (todas as adjacências examinadas).

Se $(u,v) n E$ e $u$ é preto, então $v$ já é cinza ou preto (pois todas as adjacências de $u$ foram descobertas).

In [ ]:
from collections import deque
import networkx as nx
import matplotlib.pyplot as plt

def bfs(G: nx.Graph, s):
    """
    BFS clássica em grafo não ponderado (dirigido ou não).
    Retorna: (cor, d, pai, ordem, arestas_arvore)
      - cor[v] ∈ {'white','gray','black'}
      - d[v]: distância em arestas de s até v (ou None se inalcançável)
      - pai[v]: predecessor na árvore BFS (ou None)
      - ordem: ordem de descoberta (lista)
      - arestas_arvore: lista de arestas (u,v) da árvore BFS
    """
    cor = {v: 'white' for v in G.nodes()}
    d   = {v: None for v in G.nodes()}
    pai = {v: None for v in G.nodes()}
    arestas_arvore = []
    ordem = []

    if s not in G:
        raise ValueError('Fonte s não pertence ao grafo')

    cor[s] = 'gray'
    d[s] = 0
    Q = deque([s])

    while Q:
        u = Q.popleft()
        ordem.append(u)
        # Para cada adjacente v de u
        for v in G.neighbors(u):
            if cor[v] == 'white':
                cor[v] = 'gray'
                d[v] = d[u] + 1
                pai[v] = u
                arestas_arvore.append((u, v))
                Q.append(v)
        cor[u] = 'black'

    return cor, d, pai, ordem, arestas_arvore

def reconstruir_caminho(pai, s, v):
    """Reconstrói o caminho s→v usando o dicionário de pais.
    Retorna lista de vértices ou None se v inalcançável."""
    if pai.get(v, None) is None and v != s:
        return None
    caminho = [v]
    while v != s:
        v = pai[v]
        caminho.append(v)
    caminho.reverse()
    return caminho

def arvore_bfs_subgrafo(G, arestas_arvore):
    T = nx.DiGraph()
    T.add_nodes_from(G.nodes())
    T.add_edges_from(arestas_arvore)
    return T

## 🧪 Exemplo 1: Grafo não dirigido

In [ ]:
G = nx.Graph()
G.add_edges_from([(1,2),(1,3),(2,4),(2,5),(3,6),(5,6),(6,7)])
s = 1
cor, d, pai, ordem, arestas_arvore = bfs(G, s)
print('Ordem de descoberta:', ordem)
print('Distâncias d:', d)
print('Pais π:', pai)

T = arvore_bfs_subgrafo(G, arestas_arvore)
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1000,
        font_size=12, font_weight='bold')
plt.title('Grafo original')

plt.subplot(1,2,2)
edge_colors = ['crimson' if e in T.edges() or (isinstance(G, nx.Graph) and (e[1],e[0]) in T.edges()) else 'gray' for e in G.edges()]
nx.draw(G, pos, with_labels=True, node_color='lightgreen', node_size=1000,
        font_size=12, font_weight='bold', edge_color=edge_colors, width=3)
plt.title('Arestas da árvore BFS (em vermelho)')
plt.tight_layout()
plt.show()

## 🔁 Exemplo 2: Grafo dirigido

In [ ]:
D = nx.DiGraph()
D.add_edges_from([('A','B'),('A','C'),('B','D'),('C','D'),('D','E'),('B','F')])
s = 'A'
cor, d, pai, ordem, arestas_arvore = bfs(D, s)
print('Ordem:', ordem)
print('Distâncias:', d)
print('Pais:', pai)

T = arvore_bfs_subgrafo(D, arestas_arvore)
pos = nx.spring_layout(D, seed=1)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
nx.draw(D, pos, with_labels=True, node_color='lightyellow', node_size=1000,
        font_size=12, font_weight='bold', arrows=True, arrowsize=20)
plt.title('Digrafo original')

plt.subplot(1,2,2)
nx.draw(D, pos, with_labels=True, node_color='lightgreen', node_size=1000,
        font_size=12, font_weight='bold', arrows=True, arrowsize=20)
nx.draw_networkx_edges(T, pos, edge_color='crimson', width=3, arrows=True, arrowsize=20)
plt.title('Árvore BFS (arestas em vermelho)')
plt.tight_layout()
plt.show()

## 🧭 Reconstrução de Caminhos Mínimos
Para qualquer vértice acessível $v$, a árvore BFS contém um caminho $s 	o v$ com o menor número de arestas.

In [ ]:
G = nx.Graph([(1,2),(1,3),(2,4),(2,5),(3,6),(5,6),(6,7)])
s = 1
cor, d, pai, ordem, arestas_arvore = bfs(G, s)
for v in sorted(G.nodes()):
    caminho = reconstruir_caminho(pai, s, v)
    print(f's→{v}: d={d[v]} caminho={caminho}')

# Verificar com NetworkX
print('
Verificação com shortest_path_length (NetworkX):')
for v in sorted(G.nodes()):
    print(v, nx.shortest_path_length(G, s, v))

## ⛔ Limitações (Pesos)
A BFS supõe **arestas com peso unitário** (ou custo igual). Para grafos com **pesos não negativos**, utilize **Dijkstra**; para pesos arbitrários (sem negativos), Dijkstra também; e para pesos negativos, **Bellman-Ford**.

## ⏱️ Complexidade de Tempo
- Inicialização: $O(|V|)$.
- Cada vértice entra na fila **no máximo uma vez** → total $O(|V|)$.
- Cada vértice é desenfileirado uma vez → $O(|V|)$.
- A lista de adjacência de cada vértice é percorrida uma vez → soma total $um_v 	ext{grau}(v) = 2|E| = O(|E|)$.
- Total: **$O(|V| + |E|)$**.

## 🧪 Experimento simples de custo (contadores)

In [ ]:
def bfs_com_contadores(G, s):
    from collections import deque
    cor = {v: 'white' for v in G.nodes()}
    d   = {v: None for v in G.nodes()}
    pai = {v: None for v in G.nodes()}
    Q = deque()
    enfileirar_ops = 0
    desenfileirar_ops = 0
    visitar_adj_ops = 0

    cor[s] = 'gray'; d[s] = 0; Q.append(s); enfileirar_ops += 1
    while Q:
        u = Q.popleft(); desenfileirar_ops += 1
        for v in G.neighbors(u):
            visitar_adj_ops += 1
            if cor[v] == 'white':
                cor[v] = 'gray'; d[v] = d[u] + 1; pai[v] = u
                Q.append(v); enfileirar_ops += 1
        cor[u] = 'black'
    return enfileirar_ops, desenfileirar_ops, visitar_adj_ops

G = nx.gnm_random_graph(100, 300, seed=42)
e, d, a = bfs_com_contadores(G, 0)
print('Enfileirar:', e, '| Desenfileirar:', d, '| Visitar adjacências:', a)

## 🧩 Exercícios
1. Modifique a BFS para também retornar os níveis (camadas) como listas.
2. Dado um grafo desconexo, rode BFS a partir de todo vértice não visitado e construa uma floresta BFS.
3. Use BFS para detectar se um grafo não dirigido é bipartido (2-coloração).
4. Em um digrafo, use BFS para computar todos os vértices alcançáveis a partir de $s$.
5. Compare distâncias da BFS com os comprimentos de caminhos do NetworkX em grafos aleatórios.